# Lesson 4.1 — Chart Dependencies

**Phase 4: Dependencies & Hooks**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Declare chart dependencies in `Chart.yaml`
- Download dependencies with `helm dependency update`
- Configure dependent chart values from the parent
- Conditionally enable/disable dependencies

---

## What Are Chart Dependencies?

A chart can depend on other charts. For example, a `wordpress` chart might depend on:
- `bitnami/postgresql` — the database
- `bitnami/memcached` — the cache

Instead of reinventing these components, you declare them as dependencies and let Helm manage them as part of a single release.

> **Reference:** [Chart Dependencies](https://helm.sh/docs/topics/charts/#chart-dependencies) — https://helm.sh/docs/topics/charts/#chart-dependencies

---

## Declaring Dependencies

Dependencies are declared in the `dependencies:` field of `Chart.yaml`:

In [ ]:
rm -rf /tmp/appdep && mkdir -p /tmp/appdep/templates

cat > /tmp/appdep/Chart.yaml << 'EOF'
apiVersion: v2
name: appdep
description: A chart that depends on postgresql
type: application
version: 0.1.0
appVersion: "1.0"

dependencies:
  # Depend on bitnami/postgresql
  - name: postgresql
    version: "15.x.x"   # Uses semver ranges — '15.x.x' means any 15.* patch
    repository: https://charts.bitnami.com/bitnami
    # condition: if postgresql.enabled is false in values, skip this dependency
    condition: postgresql.enabled

  # A second optional dependency (disabled by default)
  - name: redis
    version: "19.x.x"
    repository: https://charts.bitnami.com/bitnami
    condition: redis.enabled
EOF

cat /tmp/appdep/Chart.yaml

---

## Downloading Dependencies

In [ ]:
# Add the bitnami repo if not already added
helm repo add bitnami https://charts.bitnami.com/bitnami 2>/dev/null || true
helm repo update

# Download all declared dependencies into charts/
# This generates/updates Chart.lock with exact resolved versions
helm dependency update /tmp/appdep

echo ""
echo "=== Downloaded dependencies ==="
ls /tmp/appdep/charts/

In [ ]:
# Chart.lock pins exact versions — commit this file to version control!
cat /tmp/appdep/Chart.lock

> **Best Practice:** Commit `Chart.lock` to version control. It ensures reproducible builds. Run `helm dependency update` to regenerate it when changing dependency versions.

---

## Configuring Dependency Values

To pass values to a dependency, use its chart name as the top-level key in `values.yaml`:

In [ ]:
cat > /tmp/appdep/values.yaml << 'EOF'
# Our app's own values
replicaCount: 1

# postgresql subchart config — key must match the dependency 'name'
postgresql:
  enabled: true               # controls the 'condition:' in Chart.yaml
  auth:
    database: myappdb
    username: myappuser
    # IMPORTANT: Never hardcode passwords here!
    # Use an existing secret instead:
    # existingSecret: my-db-secret
    password: changeme         # only for local dev
  primary:
    persistence:
      size: 1Gi

# redis subchart config (disabled by default)
redis:
  enabled: false
  auth:
    enabled: false
EOF

cat /tmp/appdep/values.yaml

---

## The `condition` and `tags` Fields

### `condition` — Enable/Disable a Single Dependency

In [ ]:
# Add a minimal app template so there's something to render
cat > /tmp/appdep/templates/configmap.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: {{ .Release.Name }}-app-config
data:
  postgresEnabled: {{ .Values.postgresql.enabled | quote }}
  redisEnabled: {{ .Values.redis.enabled | quote }}
EOF

echo "=== With postgresql enabled (default) — should include postgresql StatefulSet ==="
helm template my-release /tmp/appdep 2>/dev/null | grep 'kind:' | sort

echo ""
echo "=== With postgresql disabled — should only render our ConfigMap ==="
helm template my-release /tmp/appdep --set postgresql.enabled=false 2>/dev/null | grep 'kind:' | sort

### `tags` — Enable/Disable Groups of Dependencies

In [ ]:
# Show a tags-based Chart.yaml example
cat << 'EOF'
# Chart.yaml with tags example
dependencies:
  - name: postgresql
    version: "15.x.x"
    repository: https://charts.bitnami.com/bitnami
    tags:
      - database       # Enable/disable by: --set tags.database=true/false
      - backend

  - name: redis
    version: "19.x.x"
    repository: https://charts.bitnami.com/bitnami
    tags:
      - cache
      - backend

# values.yaml
tags:
  backend: true    # Enables ALL charts tagged 'backend' (postgresql + redis)
  cache: false     # Disables ALL charts tagged 'cache' (redis)
EOF

---

## Dependency Management Commands

In [ ]:
echo "Dependency management commands:"
echo ""
echo "helm dependency update <chart>   # Download/refresh all deps; regenerates Chart.lock"
echo "helm dependency build <chart>    # Download deps using exact versions from Chart.lock"
echo "helm dependency list <chart>     # List all dependencies and their status"

echo ""
echo "=== List dependencies of our chart ==="
helm dependency list /tmp/appdep

---

## Installation Order

When you install a chart with dependencies, Kubernetes resources from all charts are:
1. Collected into a single set
2. Sorted by resource type (CRDs → Namespaces → RBAC → ... → Deployments)
3. Applied in that sorted order

A single `helm install` creates one release that contains resources from both the parent and all dependency charts.

---

## Exercise

In [ ]:
# TASK 1: Add a third dependency 'memcached' (bitnami/memcached, any version)
#         to /tmp/appdep/Chart.yaml, tagged with 'cache'.
#         Run helm dependency update and verify it downloads.
# TODO:


# TASK 2: Add 'memcached.enabled: false' to values.yaml,
#         then render the chart and verify memcached is not included.
# TODO:


# TASK 3: Add a template that uses the postgresql dependency's service name
#         to set a DATABASE_HOST environment variable.
#         The service name from bitnami/postgresql follows: '<release-name>-postgresql'
#         Output it as a ConfigMap value.
# TODO:

echo "Exercise: implement the tasks above"

---

## Summary

- Declare dependencies in the `dependencies:` field of `Chart.yaml`.
- `helm dependency update` downloads them to `charts/` and generates `Chart.lock`.
- Configure dependent chart values using the dependency's `name` as the top-level key in `values.yaml`.
- Use `condition:` for individual toggles, `tags:` for group toggles.
- Commit `Chart.lock` to version control for reproducible builds.

---

## Next Lesson

➡️ [Lesson 4.2 — Subcharts & Global Values](lesson-4.2-subcharts-global-values.ipynb)